In [ ]:
import math
import torch
import torch.nn as nn
import torch.nn.functional as F

torch.manual_seed(42)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(42)

device = torch.device("cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu")
print(device)

In [ ]:
# 8. Linear Attention：先算 K^T V，把 O(T^2) 注意力矩阵改为结合律计算
def linear_attention(q, k, v, eps=1e-6):
    # q/k: [B,T,D], v: [B,T,Dv]
    assert q.shape == k.shape and k.shape[:2] == v.shape[:2]
    phi_q, phi_k = F.elu(q) + 1, F.elu(k) + 1
    kv = phi_k.transpose(-2, -1) @ v                # [B,D,Dv]
    numerator = phi_q @ kv                          # [B,T,Dv]
    denominator = phi_q @ phi_k.sum(dim=-2).unsqueeze(-1)
    return numerator / denominator.clamp_min(eps)

q = torch.randn(2, 7, 8, device=device)
k = torch.randn(2, 7, 8, device=device)
v = torch.randn(2, 7, 12, device=device)
assert linear_attention(q, k, v).shape == (2, 7, 12)